In [90]:
#import necessary libraries

import pandas as pd

### CCF

In [91]:
#creating ccf dataframe and cleaning it for use in calculations
df_ccf = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/CCF.csv')
ccf_clean = df_ccf[['Carrier', 'Carbon content factor']].copy()
ccf_clean['Carbon content factor'] = pd.to_numeric(
    ccf_clean['Carbon content factor'], 
    errors='coerce'
)
ccf_clean = ccf_clean.dropna(subset=['Carbon content factor'])
ccf_clean.head()

,Carrier,Carbon content factor
1,Hard coal,0.716
2,Lignite,0.328
3,Natural gas,15.300
4,Crude oil,0.846
5,Refined oil,0.856


### Fossil inflows

In [92]:
#creating fossil inflows dataframe and cleaning it for use in calculations
# If refering to Excel, use: pd.read_excel('[name].xlsx', sheet_name='XXX')
df_fossil_inflows = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/Fossil inflows.csv')

df_fossil_inflows_clean = df_fossil_inflows.drop(columns=['Amount in dataset', 'Unit in dataset', 'Source'])
df_fossil_inflows_clean

,Flow,Added info,Amount_tonnes,Carrier,Type
0,"Crude oil, NGL, refinery feedstocks, additives...",Indigenous production,1.905703e+07,Crude oil,Inflow
1,"Crude oil, NGL, refinery feedstocks, additives...",Imports,4.970795e+08,Crude oil,Inflow
2,"Crude oil, NGL, refinery feedstocks, additives...",Exports,1.440638e+07,Crude oil,Outflow
3,"Crude oil, NGL, refinery feedstocks, additives...",Change in stock,4.042544e+06,Crude oil,Stock
4,Oil products,Imports,2.883981e+08,Refined oil,Inflow
...,...,...,...,...,...
59,Natural gas,Indigenous production,1.225769e+06,Gas,Inflow
60,Natural gas,Imports,1.252518e+07,Gas,Inflow
61,Natural gas,Exports,2.242757e+06,Gas,Outflow
62,Natural gas,Change in stock,-8.249261e+04,Gas,Stock


In [93]:
# Map and Merge so "Gas" flows find "Natural gas" factors
mapping = {
    'Gas': 'Natural gas', 
    'Crude oil': 'Crude oil', 
    'Hard coal': 'Hard coal', 
    'Lignite': 'Lignite', 
    'Refined oil': 'Refined oil'
}
df_fossil_inflows_clean['CCF_Match'] = df_fossil_inflows_clean['Carrier'].map(mapping)

# Merge the 'Carbon content factor' column to fossil data
df_final1 = df_fossil_inflows_clean.merge(ccf_clean, left_on='CCF_Match', right_on='Carrier', how='left', suffixes=('', '_ccf'))

df_final1['Amount_tonnes'] = pd.to_numeric(df_final1['Amount_tonnes'], errors='coerce')
df_final1['Carbon content factor'] = pd.to_numeric(df_final1['Carbon content factor'], errors='coerce')

# 2. Now perform the multiplication
df_final1['tonnes_Carbon'] = df_final1['Amount_tonnes'] * df_final1['Carbon content factor']


# Save the result
#df_final.to_csv('Calculated_Fossil_Carbon_Fixed.csv', index=False)
df_final1.info()
df_final1.tail()


<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Flow                   64 non-null     str    
 1   Added info             64 non-null     str    
 2   Amount_tonnes          64 non-null     float64
 3   Carrier                64 non-null     str    
 4   Type                   64 non-null     str    
 5   CCF_Match              64 non-null     str    
 6   Carrier_ccf            64 non-null     str    
 7   Carbon content factor  64 non-null     float64
 8   tonnes_Carbon          64 non-null     float64
dtypes: float64(3), str(6)
memory usage: 4.6 KB


,Flow,Added info,Amount_tonnes,Carrier,Type,CCF_Match,Carrier_ccf,Carbon content factor,tonnes_Carbon
59,Natural gas,Indigenous production,1.225769e+06,Gas,Inflow,Natural gas,Natural gas,15.3,1.875427e+07
60,Natural gas,Imports,1.252518e+07,Gas,Inflow,Natural gas,Natural gas,15.3,1.916352e+08
61,Natural gas,Exports,2.242757e+06,Gas,Outflow,Natural gas,Natural gas,15.3,3.431418e+07
62,Natural gas,Change in stock,-8.249261e+04,Gas,Stock,Natural gas,Natural gas,15.3,-1.262137e+06
63,Natural gas,International maritime bunkers,1.819346e+04,Gas,Outflow,Natural gas,Natural gas,15.3,2.783600e+05


In [94]:
primary_fuels = [
    "Crude oil, NGL, refinery feedstocks, additives and oxygenates and other hydrocarbons",
    "Natural gas",
    "Hard coal",
    "Brown coal",
    "Oil shale and oil sands",
    "Peat"
]

df_primary_fossil = df_final1[df_final1["Flow"].isin(primary_fuels)]



In [95]:
def fossil_supply_calc(group):

    production = group.loc[group["Added info"] == "Indigenous production", "tonnes_Carbon"].sum()
    imports = group.loc[group["Added info"] == "Imports", "tonnes_Carbon"].sum()
    exports = group.loc[group["Added info"] == "Exports", "tonnes_Carbon"].sum()
    stock = group.loc[group["Added info"] == "Change in stock", "tonnes_Carbon"].sum()
    bunkers = group.loc[group["Added info"] == "International maritime bunkers", "tonnes_Carbon"].sum()

    return production + imports - exports - bunkers - stock


fossil_supply_by_fuel = df_primary_fossil.groupby("Flow").apply(fossil_supply_calc)
fossil_supply_by_fuel



Flow
Brown coal                                                                              1.620279e+08
Crude oil, NGL, refinery feedstocks, additives and oxygenates and other hydrocarbons    4.210437e+08
Hard coal                                                                               9.691252e+07
Natural gas                                                                             1.770591e+08
Oil shale and oil sands                                                                 1.249171e+07
Peat                                                                                    2.815183e+05
dtype: float64

In [96]:
fossil_supply_df = fossil_supply_by_fuel.reset_index()
fossil_supply_df.columns = ["Flow", "Fossil_supply_tC"]

fossil_supply_df
#fossil_supply_df["Type"] = "Fossil inflow"
fossil_supply_df.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Aggregated_Fossil_Carbon_Inflow.csv', index=False)

In [97]:
df_final_fossil_inflow = df_final.drop(columns=['Amount_tonnes', 'CCF_Match', 'Carrier_ccf', 'Carbon content factor [kg C/kg]'])
df_final_fossil_inflow.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Calculated_Fossil_Carbon_Inflow.csv', index=False)

### GHG

In [98]:
#creating ghg outflows dataframe and cleaning it for use in calculations
# If refering to Excel, use: pd.read_excel('[name].xlsx', sheet_name='XXX')
df_ghg_outflows = pd.read_csv('/Users/charlottewestenberg/Thesis/00data/from_excel/_PC_ghg_emissions.csv')
df_ghg_outflows.columns


Index(['Flow', 'Added info', 'Unit in dataset', '#CO2 in dataset',
       'Amount_tonnes_CO2', '#CH4 in dataset', 'Amount_tonnes_CH4',
       '#CO in dataset', 'Amount_tonnes_CO', 'Sankey node flow name', 'Type',
       'Source'],
      dtype='str')

In [99]:
df_ghg_outflows_clean = df_ghg_outflows.drop(columns=['Unit in dataset', '#CO2 in dataset', '#CH4 in dataset', '#CO in dataset', 'Source'])
df_ghg_outflows_clean.info()
df_ghg_outflows_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Flow                   11 non-null     str    
 1   Added info             27 non-null     str    
 2   Amount_tonnes_CO2      31 non-null     float64
 3   Amount_tonnes_CH4      31 non-null     float64
 4   Amount_tonnes_CO       31 non-null     float64
 5   Sankey node flow name  31 non-null     str    
 6   Type                   31 non-null     str    
dtypes: float64(3), str(4)
memory usage: 4.1 KB


,Flow,Added info,Amount_tonnes_CO2,Amount_tonnes_CH4,Amount_tonnes_CO,Sankey node flow name,Type
0,1.A.1. Energy industries,1.A.1.a. Public electricity and heat production,5.679841e+08,119356.697800,283681.48470,Energy production,Outflow
1,NaN,1.A.1.b. Petroleum refining,9.290136e+07,2359.067881,17549.60913,Energy,Outflow
2,NaN,1.A.1.c. Manufacture of solid fuels and other ...,2.765303e+07,6357.444519,82132.55458,Energy,Outflow
3,1.A.2. Manufacturing industries and construction,1.A.2.a. Iron and steel,6.759795e+07,4597.164908,613217.15560,Manufacturing,Outflow
4,NaN,1.A.2.b. Non-ferrous metals,7.721146e+06,391.923416,12262.71470,Manufacturing,Outflow


In [100]:
# dropping ghost rows from the old excel file (all the 31 nan rows)

df_ghg_outflows_clean = df_ghg_outflows_clean.dropna(subset=['Sankey node flow name'])

df_ghg_outflows_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Flow                   11 non-null     str    
 1   Added info             27 non-null     str    
 2   Amount_tonnes_CO2      31 non-null     float64
 3   Amount_tonnes_CH4      31 non-null     float64
 4   Amount_tonnes_CO       31 non-null     float64
 5   Sankey node flow name  31 non-null     str    
 6   Type                   31 non-null     str    
dtypes: float64(3), str(4)
memory usage: 1.8 KB


In [101]:
ccf_library = df_ccf.dropna(subset=['Carbon content factor [kg C/kg]'])
ccf_library = dict(zip(ccf_library['Carrier'], ccf_library['Carbon content factor [kg C/kg]']))

# Print it to see your "Library"
print("Your Automated CCF Library:")
for carrier, value in ccf_library.items():
    print(f" - {carrier}: {value} kg C/kg")

KeyError: ['Carbon content factor [kg C/kg]']

In [ ]:
CCF_FACTORS = {
    'Amount_tonnes_CO2': 0.27,
    'Amount_tonnes_CH4': 0.75,
    'Amount_tonnes_CO': 0.4286
}
		

# 3. Clean and Multiply in one loop
# This ensures columns are numeric and then creates new 'Carbon' columns
for gas, factor in CCF_FACTORS.items():
    if gas in df_ghg_outflows_clean.columns:
        df_ghg_outflows_clean[gas] = pd.to_numeric(df_ghg_outflows_clean[gas], errors='coerce').fillna(0)
        new_col_name = f'Carbon_from_{gas}'
        df_ghg_outflows_clean[new_col_name] = df_ghg_outflows_clean[gas] * factor

# 4. Total Carbon from all GHGs
# This sums up the new columns to give you the total carbon outflow
carbon_cols = [f'Carbon_from_{gas}' for gas in CCF_FACTORS.keys() if gas in df_ghg_outflows_clean.columns]
df_ghg_outflows_clean['Total_Carbon_GHG'] = df_ghg_outflows_clean[carbon_cols].sum(axis=1)

df_ghg_outflows_clean.head()

# Save result
#df_ghg_outflows_clean.to_csv('Calculated_GHG_Carbon.csv', index=False)

,Flow,Added info,Amount_tonnes_CO2,Amount_tonnes_CH4,Amount_tonnes_CO,Sankey node flow name,Type,Carbon_from_Amount_tonnes_CO2,Carbon_from_Amount_tonnes_CH4,Carbon_from_Amount_tonnes_CO,Total_Carbon_GHG
0,1.A.1. Energy industries,1.A.1.a. Public electricity and heat production,5.679841e+08,119356.697800,283681.48470,Energy production,Outflow,1.533557e+08,89517.523350,121585.884342,1.535668e+08
1,NaN,1.A.1.b. Petroleum refining,9.290136e+07,2359.067881,17549.60913,Energy,Outflow,2.508337e+07,1769.300911,7521.762473,2.509266e+07
2,NaN,1.A.1.c. Manufacture of solid fuels and other ...,2.765303e+07,6357.444519,82132.55458,Energy,Outflow,7.466318e+06,4768.083389,35202.012893,7.506288e+06
3,1.A.2. Manufacturing industries and construction,1.A.2.a. Iron and steel,6.759795e+07,4597.164908,613217.15560,Manufacturing,Outflow,1.825145e+07,3447.873681,262824.872890,1.851772e+07
4,NaN,1.A.2.b. Non-ferrous metals,7.721146e+06,391.923416,12262.71470,Manufacturing,Outflow,2.084709e+06,293.942562,5255.799520,2.090259e+06


In [ ]:
df_ghg_outflows_clean.drop(columns=['Amount_tonnes_CO2', 'Amount_tonnes_CH4', 'Amount_tonnes_CO'], inplace=True)
df_ghg_outflows_clean = df_ghg_outflows_clean.rename(columns={
    'Carbon_from_Amount_tonnes_CO2': 'Carbon_from_CO2',
    'Carbon_from_Amount_tonnes_CH4': 'Carbon_from_CH4',
    'Carbon_from_Amount_tonnes_CO': 'Carbon_from_CO'
    })

In [ ]:
df_ghg_outflows_clean.to_csv('/Users/charlottewestenberg/Thesis/00data/processed/Calculated_GHG_Carbon_Outflow.csv', index=False)